In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
columns = [
    'Class',
    'Alcohol',
    'Malic_acid',
    'Ash',
    'Alcalinity_of_ash',
    'Magnesium',
    'Total_phenols',
    'Flavanoids',
    'Nonflavanoid_phenols',
    'Proanthocyanins',
    'Color_intensity',
    'Hue',
    'OD280/OD315_of_diluted_wines',
    'Proline'
]


In [5]:
df = pd.read_csv(r"D:\Learn_ML\ML_algos\datasets\Wine\wine.data" , header = None , names = columns)
df.head()

,Class,Alcohol,Malic_acid,Ash,Alcalinity_of_ash,Magnesium,Total_phenols,Flavanoids,Nonflavanoid_phenols,Proanthocyanins,Color_intensity,Hue,OD280/OD315_of_diluted_wines,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


In [6]:
df.isnull().sum()

Class                           0
Alcohol                         0
Malic_acid                      0
Ash                             0
Alcalinity_of_ash               0
Magnesium                       0
Total_phenols                   0
Flavanoids                      0
Nonflavanoid_phenols            0
Proanthocyanins                 0
Color_intensity                 0
Hue                             0
OD280/OD315_of_diluted_wines    0
Proline                         0
dtype: int64

In [10]:
y = df["Class"]
X = df.drop(columns = ['Class'])
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , stratify = y)
print(X_train.shape , y_train.shape , X_test.shape , y_test.shape)

(142, 13) (142,) (36, 13) (36,)


In [14]:
print(len(y_train.value_counts()))

3


In [43]:
X_train['Alcohol'][0]

np.float64(14.23)

In [11]:
class TreeNode:
    def __init__(self , feature = None , threshold = None , left = None , right = None , value = None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


In [40]:
def calculate_gini(y):
   y_value_counts = y.value_counts()
   total_y = len(y)
   gini = 0
   for cls , count_per_cls in y_value_counts.items():
      prob = count_per_cls / total_y
      gini += (prob**2)
   return (1 - gini)

In [41]:
calculate_gini(y_train)

0.6577068042055148

In [44]:
def build_tree(X, y, depth, max_depth=5):
    if y.nunique() == 1:
        return TreeNode(value=y.iloc[0])
    if depth >= max_depth:
        return TreeNode(value=y.value_counts().idxmax())
    parent_gini = calculate_gini(y)
    max_info_gain = float('-inf')
    feature = None
    threshold = None
    for col in X:
        column_values = X[col].unique()
        sorted_values = np.sort(column_values)
        median_splits = []
        for i in range(1, len(sorted_values)):
            midpoint = (sorted_values[i - 1] + sorted_values[i]) / 2
            median_splits.append(midpoint)
        for val in median_splits:
            mask = X[col] <= val
            y_left = y[mask]
            y_right = y[~mask]
            if len(y_left) == 0 or len(y_right) == 0:
                continue
            gini_left = calculate_gini(y_left)
            gini_right = calculate_gini(y_right)

            weighted_gini = (
                (len(y_left) / len(y)) * gini_left
                + (len(y_right) / len(y)) * gini_right
            )
            information_gain = parent_gini - weighted_gini
            if information_gain > max_info_gain:
                max_info_gain = information_gain
                feature = col
                threshold = val
    if feature is None or max_info_gain <= 0:
        return TreeNode(value=y.value_counts().idxmax())
    left_mask = X[feature] <= threshold
    right_mask = X[feature] > threshold

    left_subtree = build_tree(X[left_mask], y[left_mask], depth + 1, max_depth)
    right_subtree = build_tree(X[right_mask], y[right_mask], depth + 1, max_depth)
    return TreeNode(
        feature=feature,
        threshold=threshold,
        left=left_subtree,
        right=right_subtree,
        value=None
    )


In [47]:
def predict_one(node, x):
    if node.value is not None:
        return node.value
    if x[node.feature] <= node.threshold:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)


In [48]:
def predict(tree, X):
    predictions = []
    for i in range(len(X)):
        x = X.iloc[i]
        pred = predict_one(tree, x)
        predictions.append(pred)
    return predictions


In [49]:
tree = build_tree(X_train, y_train, depth=0, max_depth=100)
y_pred = predict(tree, X_test)


In [50]:
results_df = X_test.copy()
results_df["Actual"] = y_test.values
results_df["Predicted"] = y_pred


In [51]:
results_df

,Alcohol,Malic_acid,Ash,Alcalinity_of_ash,Magnesium,Total_phenols,Flavanoids,Nonflavanoid_phenols,Proanthocyanins,Color_intensity,Hue,OD280/OD315_of_diluted_wines,Proline,Actual,Predicted
174,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.300000,0.70,1.56,750,3,3
13,14.75,1.73,2.39,11.4,91,3.10,3.69,0.43,2.81,5.400000,1.25,2.73,1150,1,1
93,12.29,2.83,2.22,18.0,88,2.45,2.25,0.25,1.99,2.150000,1.15,3.30,290,2,2
20,14.06,1.63,2.28,16.0,126,3.00,3.17,0.24,2.10,5.650000,1.09,3.71,780,1,1
7,14.06,2.15,2.61,17.6,121,2.60,2.51,0.31,1.25,5.050000,1.06,3.58,1295,1,1
54,13.74,1.67,2.25,16.4,118,2.60,2.90,0.21,1.62,5.850000,0.92,3.20,1060,1,1
123,13.05,5.80,2.13,21.5,86,2.62,2.65,0.30,2.01,2.600000,0.73,3.10,380,2,2
56,14.22,1.70,2.30,16.3,118,3.20,3.00,0.26,2.03,6.380000,0.94,3.31,970,1,1
75,11.66,1.88,1.92,16.0,97,1.61,1.57,0.34,1.15,3.800000,1.23,2.14,428,2,2
160,12.36,3.83,2.38,21.0,88,2.30,0.92,0.50,1.04,7.650000,0.56,1.58,520,3,3
